In [40]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Load dataset
df = pd.read_csv("../data/raw/task_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Dataset shape: (10000, 10)


,Task_ID,Task_Description,Category,Priority,Assigned_To,Status,Estimated_Hours,Completed_Hours,Created_Date,Deadline
0,TASK_00001,Add input validation security,Security,Critical,Alice,On Hold,19,12,2026-07-08,2026-07-22
1,TASK_00002,Develop team management page,Frontend,Medium,Alice,On Hold,11,5,2026-03-31,2026-04-09
2,TASK_00003,Create responsive frontend,Frontend,Low,Frank,Pending,1,0,2026-06-04,2026-06-19
3,TASK_00004,Deploy application to server,DevOps,High,Frank,In Progress,2,1,2026-07-02,2026-07-22
4,TASK_00005,Create backend validation,Backend,Medium,Alice,In Progress,1,0,2026-03-30,2026-04-21


In [41]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
Task_ID             0
Task_Description    0
Category            0
Priority            0
Assigned_To         0
Status              0
Estimated_Hours     0
Completed_Hours     0
Created_Date        0
Deadline            0
dtype: int64

Duplicate rows: 0


In [42]:
# Convert task descriptions to string
df["Task_Description"] = df["Task_Description"].astype(str)

# Convert text to lowercase
df["Cleaned_Task_Description"] = df["Task_Description"].str.lower()

df[["Task_Description", "Cleaned_Task_Description"]].head(10)

,Task_Description,Cleaned_Task_Description
0,Add input validation security,add input validation security
1,Develop team management page,develop team management page
2,Create responsive frontend,create responsive frontend
3,Deploy application to server,deploy application to server
4,Create backend validation,create backend validation
5,Update README documentation,update readme documentation
6,Fix authorization issue,fix authorization issue
7,Implement task workflow service,implement task workflow service
8,Create task assignment service,create task assignment service
9,Write unit tests for authentication,write unit tests for authentication


In [43]:
def remove_special_characters(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

df["Cleaned_Task_Description"] = (
    df["Cleaned_Task_Description"]
    .apply(remove_special_characters)
)

df[["Task_Description", "Cleaned_Task_Description"]].head(10)

,Task_Description,Cleaned_Task_Description
0,Add input validation security,add input validation security
1,Develop team management page,develop team management page
2,Create responsive frontend,create responsive frontend
3,Deploy application to server,deploy application to server
4,Create backend validation,create backend validation
5,Update README documentation,update readme documentation
6,Fix authorization issue,fix authorization issue
7,Implement task workflow service,implement task workflow service
8,Create task assignment service,create task assignment service
9,Write unit tests for authentication,write unit tests for authentication


In [44]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    return " ".join(filtered_words)

df["Cleaned_Task_Description"] = (
    df["Cleaned_Task_Description"]
    .apply(remove_stopwords)
)

df[["Task_Description", "Cleaned_Task_Description"]].head(10)

,Task_Description,Cleaned_Task_Description
0,Add input validation security,add input validation security
1,Develop team management page,develop team management page
2,Create responsive frontend,create responsive frontend
3,Deploy application to server,deploy application server
4,Create backend validation,create backend validation
5,Update README documentation,update readme documentation
6,Fix authorization issue,fix authorization issue
7,Implement task workflow service,implement task workflow service
8,Create task assignment service,create task assignment service
9,Write unit tests for authentication,write unit tests authentication


In [45]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]
    return " ".join(lemmatized_words)

df["Cleaned_Task_Description"] = (
    df["Cleaned_Task_Description"]
    .apply(lemmatize_text)
)

df[["Task_Description", "Cleaned_Task_Description"]].head(10)

,Task_Description,Cleaned_Task_Description
0,Add input validation security,add input validation security
1,Develop team management page,develop team management page
2,Create responsive frontend,create responsive frontend
3,Deploy application to server,deploy application server
4,Create backend validation,create backend validation
5,Update README documentation,update readme documentation
6,Fix authorization issue,fix authorization issue
7,Implement task workflow service,implement task workflow service
8,Create task assignment service,create task assignment service
9,Write unit tests for authentication,write unit test authentication


In [46]:
df[
    [
        "Task_ID",
        "Task_Description",
        "Cleaned_Task_Description",
        "Category",
        "Priority"
    ]
].head(10)

,Task_ID,Task_Description,Cleaned_Task_Description,Category,Priority
0,TASK_00001,Add input validation security,add input validation security,Security,Critical
1,TASK_00002,Develop team management page,develop team management page,Frontend,Medium
2,TASK_00003,Create responsive frontend,create responsive frontend,Frontend,Low
3,TASK_00004,Deploy application to server,deploy application server,DevOps,High
4,TASK_00005,Create backend validation,create backend validation,Backend,Medium
5,TASK_00006,Update README documentation,update readme documentation,Documentation,Medium
6,TASK_00007,Fix authorization issue,fix authorization issue,Security,High
7,TASK_00008,Implement task workflow service,implement task workflow service,Backend,Low
8,TASK_00009,Create task assignment service,create task assignment service,Backend,High
9,TASK_00010,Write unit tests for authentication,write unit test authentication,Testing,High


In [47]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/task_dataset_cleaned.csv"

df.to_csv(output_path, index=False)

print("Processed dataset saved successfully!")
print(output_path)

Processed dataset saved successfully!
../data/processed/task_dataset_cleaned.csv


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Transform cleaned task descriptions
X_tfidf = tfidf.fit_transform(df["Cleaned_Task_Description"])

print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (10000, 299)


In [49]:
from sklearn.model_selection import train_test_split

X = X_tfidf
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 8000
Testing samples: 2000
